# Tarea E — Regularización Ridge y validación cruzada sin tocar la prueba

## Parte 1 — Datos

In [1]:
# Subtarea T1: Cargar sklearn.datasets.load_diabetes, reportar número de ejemplos y de atributos, y dividir en entrenamiento/prueba 75%/25% con train_test_split(rand
# SUBTAREA T1: Cargar el conjunto Diabetes, reportar dimensiones y hacer la
# única división entrenamiento/prueba de la tarea (75%/25%, random_state=7).
# El conjunto de prueba queda reservado para la Parte 3.

import json
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split


def main():
    # ------------------------------------------------------------------
    # 1) Cargar los datos
    # ------------------------------------------------------------------
    data = load_diabetes()
    X = data.data
    y = data.target

    n_ejemplos = int(X.shape[0])
    n_atributos = int(X.shape[1])

    # ------------------------------------------------------------------
    # 2) División única 75% / 25% con random_state=7
    #    (estratificación no aplica: es un problema de regresión)
    # ------------------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=7
    )

    n_train = int(X_train.shape[0])
    n_test = int(X_test.shape[0])
    prop_train = float(n_train / n_ejemplos)
    prop_test = float(n_test / n_ejemplos)

    # Estadísticas descriptivas del objetivo (solo informativas; el conjunto
    # de prueba NO se usa para nada más en esta subtarea)
    media_y_total = float(np.mean(y))
    desv_y_total = float(np.std(y))
    media_y_train = float(np.mean(y_train))
    desv_y_train = float(np.std(y_train))

    # ------------------------------------------------------------------
    # 3) Guardar todas las cifras calculadas en resultados.json
    # ------------------------------------------------------------------
    resultados = {
        "subtarea": "T1_carga_y_division_diabetes",
        "dataset": "sklearn.datasets.load_diabetes",
        "n_ejemplos_total": n_ejemplos,
        "n_atributos": n_atributos,
        "n_ejemplos_entrenamiento": n_train,
        "n_ejemplos_prueba": n_test,
        "proporcion_entrenamiento": prop_train,
        "proporcion_prueba": prop_test,
        "test_size": 0.25,
        "random_state": 7,
        "media_objetivo_total": media_y_total,
        "desv_est_objetivo_total": desv_y_total,
        "media_objetivo_entrenamiento": media_y_train,
        "desv_est_objetivo_entrenamiento": desv_y_train,
    }

    with open("resultados.json", "w") as f:
        json.dump(resultados, f, indent=2)

    # ------------------------------------------------------------------
    # 4) Reporte de las cifras principales con 4 decimales
    # ------------------------------------------------------------------
    print("=== T1: Carga y división del conjunto Diabetes ===")
    print(f"Ejemplos totales: {n_ejemplos}")
    print(f"Atributos: {n_atributos}")
    print(f"Entrenamiento: {n_train} ejemplos ({prop_train * 100:.4f} %)")
    print(f"Prueba: {n_test} ejemplos ({prop_test * 100:.4f} %)")
    print(f"Media del objetivo (total): {media_y_total:.4f}")
    print(f"Desv. est. del objetivo (total): {desv_y_total:.4f}")
    print(f"Media del objetivo (entrenamiento): {media_y_train:.4f}")
    print(f"Desv. est. del objetivo (entrenamiento): {desv_y_train:.4f}")
    print("Resultados guardados en resultados.json")
    print("El conjunto de prueba queda reservado para la Parte 3.")


if __name__ == "__main__":
    main()


=== T1: Carga y división del conjunto Diabetes ===
Ejemplos totales: 442
Atributos: 10
Entrenamiento: 331 ejemplos (74.8869 %)
Prueba: 111 ejemplos (25.1131 %)
Media del objetivo (total): 152.1335
Desv. est. del objetivo (total): 77.0057
Media del objetivo (entrenamiento): 152.4290
Desv. est. del objetivo (entrenamiento): 76.9385
Resultados guardados en resultados.json
El conjunto de prueba queda reservado para la Parte 3.


El conjunto Diabetes que trae scikit-learn tiene 442 ejemplos y 10 atributos. Lo dividí en entrenamiento y prueba con test_size = 0.25 y random_state = 7, lo que dejó 331 ejemplos para entrenamiento (74.8869 %) y 111 para prueba (25.1131 %). Esta es la única división de la tarea: el conjunto de prueba queda reservado y se usa una sola vez, al final de la Parte 3. La celda de código muestra además la media y la desviación estándar del objetivo (152.1335 y 77.0057 en los 442 ejemplos; 152.4290 y 76.9385 en entrenamiento), valores casi iguales entre sí, señal de que la división no deformó la distribución de la variable respuesta.

## Parte 2 — Selección de alpha por validación

In [2]:
# Subtarea T2: Con solo el conjunto de entrenamiento: pipeline StandardScaler+Ridge, GridSearchCV sobre alpha=numpy.logspace(-3,3,13) con KFold(n_splits=5, shuffle=T
import json
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


def main():
    # ---------- Datos y división idéntica a la subtarea T1 ----------
    data = load_diabetes()
    X, y = data.data, data.target
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=7
    )

    # ---------- Pipeline StandardScaler + Ridge ----------
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge()),
    ])

    # ---------- Rejilla de alphas y esquema de validación cruzada ----------
    alphas = np.logspace(-3, 3, 13)
    param_grid = {"ridge__alpha": alphas}
    cv = KFold(n_splits=5, shuffle=True, random_state=7)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="r2",
        cv=cv,
        return_train_score=False,
    )

    # Se ajusta SOLO con el conjunto de entrenamiento
    grid.fit(X_train, y_train)

    # ---------- Cifras a reportar ----------
    best_index = int(grid.best_index_)
    best_alpha = float(grid.best_params_["ridge__alpha"])
    best_cv_r2 = float(grid.best_score_)

    alphas_list = [float(a) for a in alphas]
    mean_scores = [float(s) for s in grid.cv_results_["mean_test_score"]]
    std_scores = [float(s) for s in grid.cv_results_["std_test_score"]]

    # R2 de cada fold para el alpha elegido
    split_keys = [
        k for k in grid.cv_results_
        if k.startswith("split") and k.endswith("_test_score")
    ]
    split_keys_sorted = sorted(
        split_keys, key=lambda s: int(s.split("_")[0].replace("split", ""))
    )
    best_fold_scores = [
        float(grid.cv_results_[k][best_index]) for k in split_keys_sorted
    ]

    resultados = {
        "subtarea": "T2_seleccion_alpha_validacion_cruzada",
        "dataset": "sklearn.datasets.load_diabetes",
        "n_ejemplos_entrenamiento": int(X_train.shape[0]),
        "n_ejemplos_prueba_reservados": int(X_test.shape[0]),
        "pipeline": "StandardScaler + Ridge",
        "rejilla_alpha": "numpy.logspace(-3, 3, 13)",
        "cv": "KFold(n_splits=5, shuffle=True, random_state=7)",
        "scoring": "r2",
        "n_alphas_evaluados": len(alphas_list),
        "alphas_evaluados": alphas_list,
        "r2_medio_cv_por_alpha": mean_scores,
        "r2_std_cv_por_alpha": std_scores,
        "indice_mejor_alpha": best_index,
        "alpha_elegido": best_alpha,
        "r2_medio_cv_alpha_elegido": best_cv_r2,
        "r2_std_cv_alpha_elegido": std_scores[best_index],
        "r2_por_fold_alpha_elegido": best_fold_scores,
    }

    # ---------- Guardar en resultados.json (conservando subtareas previas) ----------
    previo = {}
    try:
        with open("resultados.json", "r") as f:
            cargado = json.load(f)
        if isinstance(cargado, dict):
            previo = dict(cargado)
    except Exception:
        previo = {}

    if "resultados" in previo and isinstance(previo["resultados"], dict):
        nombre_previo = previo["resultados"].get("subtarea", "subtarea_anterior")
        previo[nombre_previo] = previo.pop("resultados")

    previo["resultados"] = resultados
    with open("resultados.json", "w") as f:
        json.dump(previo, f, indent=2)

    # ---------- Impresión de cifras principales (4 decimales) ----------
    print("=== T2: Selección de alpha por validación cruzada (solo entrenamiento) ===")
    print(f"Entrenamiento: {X_train.shape[0]} ejemplos | Prueba reservada (no usada aquí): {X_test.shape[0]}")
    print(f"Alphas evaluados: {len(alphas_list)} valores de numpy.logspace(-3, 3, 13)")
    print(f"CV: KFold(n_splits=5, shuffle=True, random_state=7) | scoring: r2")
    print("-" * 52)
    print(f"{'Alpha':>12} {'R2 medio CV':>14} {'R2 std CV':>12}")
    for a, m, s in zip(alphas_list, mean_scores, std_scores):
        print(f"{a:12.4f} {m:14.4f} {s:12.4f}")
    print("-" * 52)
    print(f"Alpha elegido: {best_alpha:.4f}")
    print(f"R2 medio de validación cruzada: {best_cv_r2:.4f}")
    print(f"Desv. est. del R2 en CV (alpha elegido): {std_scores[best_index]:.4f}")
    print("R2 por fold (alpha elegido): " + ", ".join(f"{s:.4f}" for s in best_fold_scores))
    print("Resultados guardados en resultados.json")


if __name__ == "__main__":
    main()


=== T2: Selección de alpha por validación cruzada (solo entrenamiento) ===
Entrenamiento: 331 ejemplos | Prueba reservada (no usada aquí): 111
Alphas evaluados: 13 valores de numpy.logspace(-3, 3, 13)
CV: KFold(n_splits=5, shuffle=True, random_state=7) | scoring: r2
----------------------------------------------------
       Alpha    R2 medio CV    R2 std CV
      0.0010         0.4724       0.0703
      0.0032         0.4724       0.0703
      0.0100         0.4724       0.0703
      0.0316         0.4724       0.0703
      0.1000         0.4724       0.0705
      0.3162         0.4724       0.0709
      1.0000         0.4722       0.0717
      3.1623         0.4714       0.0730
     10.0000         0.4708       0.0739
     31.6228         0.4698       0.0741
    100.0000         0.4570       0.0732
    316.2278         0.3995       0.0683
   1000.0000         0.2790       0.0514
----------------------------------------------------
Alpha elegido: 0.3162
R2 medio de validación cruzada:

Ajusté un pipeline StandardScaler + Ridge y busqué alpha entre los 13 valores de numpy.logspace(-3, 3, 13) con GridSearchCV, KFold(n_splits=5, shuffle=True, random_state=7) y scoring = "r2", usando únicamente los 331 ejemplos de entrenamiento. El alpha elegido fue 0.3162, con un R² medio de validación cruzada de 0.4724 y desviación estándar entre folds de 0.0709 (los R² por fold fueron 0.5643, 0.3874, 0.5356, 0.3987 y 0.4762). La tabla con los 13 alphas, sus R² medios y sus desviaciones estándar queda en la salida de la celda de código.

## Parte 3 — Evaluación en prueba

In [3]:
# Subtarea T3: Con el alpha elegido en T2, evaluar una sola vez sobre el conjunto de prueba (25%, random_state=7): reportar R2 y RMSE del pipeline Ridge y comparar c
# -*- coding: utf-8 -*-
"""
SUBTAREA T3: Evaluación única sobre el conjunto de prueba.
- Recalcula de forma independiente la división 75/25 (random_state=7) y la selección
  de alpha por validación cruzada (para reproducir el alpha elegido en T2).
- Con ese alpha, evalúa UNA sola vez el pipeline StandardScaler+Ridge sobre la prueba:
  reporta R2 y RMSE.
- Compara con LinearRegression (mismo StandardScaler) evaluada sobre la misma prueba.
- Sin figuras. Guarda todas las cifras en resultados.json.
"""

import json
import math

import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# ---------------------------------------------------------------------------
# 1) Datos y división idéntica a T1 (test 25%, random_state=7)
# ---------------------------------------------------------------------------
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# ---------------------------------------------------------------------------
# 2) Reproducción de la selección de alpha de T2 (SOLO entrenamiento)
#    Rejilla: numpy.logspace(-3, 3, 13); CV: KFold(5, shuffle=True, random_state=7)
# ---------------------------------------------------------------------------
alphas = np.logspace(-3, 3, 13)
pipe_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge()),
])
cv = KFold(n_splits=5, shuffle=True, random_state=7)
grid = GridSearchCV(
    estimator=pipe_ridge,
    param_grid={"ridge__alpha": alphas},
    cv=cv,
    scoring="r2",
)
grid.fit(X_train, y_train)  # solo entrenamiento: la prueba no participa aquí

alpha_recalculado = float(grid.best_params_["ridge__alpha"])
r2_cv_alpha = float(grid.best_score_)

# Alpha elegido en T2 (referencia del enunciado); verificamos que la reproducción coincide
alpha_t2 = 0.31622776601683794
coincide = math.isclose(alpha_recalculado, alpha_t2, rel_tol=1e-12, abs_tol=1e-12)
alpha_final = alpha_t2 if coincide else alpha_recalculado

# ---------------------------------------------------------------------------
# 3) Evaluación ÚNICA sobre el conjunto de prueba: Ridge con el alpha elegido
# ---------------------------------------------------------------------------
pipe_ridge_final = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=alpha_final)),
])
pipe_ridge_final.fit(X_train, y_train)
y_pred_ridge = pipe_ridge_final.predict(X_test)

r2_prueba_ridge = float(r2_score(y_test, y_pred_ridge))
rmse_prueba_ridge = float(np.sqrt(mean_squared_error(y_test, y_pred_ridge)))

# ---------------------------------------------------------------------------
# 4) Comparación: LinearRegression con el mismo StandardScaler, misma prueba
# ---------------------------------------------------------------------------
pipe_linreg = Pipeline([
    ("scaler", StandardScaler()),
    ("linreg", LinearRegression()),
])
pipe_linreg.fit(X_train, y_train)
y_pred_linreg = pipe_linreg.predict(X_test)

r2_prueba_linreg = float(r2_score(y_test, y_pred_linreg))
rmse_prueba_linreg = float(np.sqrt(mean_squared_error(y_test, y_pred_linreg)))

# Diferencias (Ridge - Lineal)
dif_r2 = float(r2_prueba_ridge - r2_prueba_linreg)
dif_rmse = float(rmse_prueba_ridge - rmse_prueba_linreg)

mejor_modelo = "Ridge" if r2_prueba_ridge >= r2_prueba_linreg else "LinearRegression"

# ---------------------------------------------------------------------------
# 5) Guardado de todas las cifras en resultados.json
# ---------------------------------------------------------------------------
resultados = {
    "subtarea": "T3_evaluacion_en_prueba",
    "dataset": "sklearn.datasets.load_diabetes",
    "test_size": 0.25,
    "random_state": 7,
    "n_ejemplos_entrenamiento": int(X_train.shape[0]),
    "n_ejemplos_prueba": int(X_test.shape[0]),
    "pipeline_ridge": "StandardScaler + Ridge",
    "pipeline_lineal": "StandardScaler + LinearRegression",
    "rejilla_alpha": "numpy.logspace(-3, 3, 13)",
    "cv": "KFold(n_splits=5, shuffle=True, random_state=7)",
    "scoring": "r2",
    "alpha_elegido_T2": float(alpha_t2),
    "alpha_recalculado_grid": float(alpha_recalculado),
    "alpha_recalculado_coincide_con_T2": bool(coincide),
    "alpha_usado_en_prueba": float(alpha_final),
    "r2_medio_cv_alpha_elegido": float(r2_cv_alpha),
    "r2_prueba_ridge": r2_prueba_ridge,
    "rmse_prueba_ridge": rmse_prueba_ridge,
    "r2_prueba_lineal": r2_prueba_linreg,
    "rmse_prueba_lineal": rmse_prueba_linreg,
    "diferencia_r2_ridge_menos_lineal": dif_r2,
    "diferencia_rmse_ridge_menos_lineal": dif_rmse,
    "mejor_modelo_en_prueba": mejor_modelo,
}

with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

# ---------------------------------------------------------------------------
# 6) Impresión de las cifras principales (4 decimales)
# ---------------------------------------------------------------------------
print("=== T3: Evaluación única sobre el conjunto de prueba ===")
print(f"Entrenamiento: {X_train.shape[0]} ejemplos | Prueba: {X_test.shape[0]} ejemplos (test_size=0.25, random_state=7)")
print("La prueba se usa una sola vez, con el alpha ya elegido en T2 (solo entrenamiento/CV).")
print("----------------------------------------------------")
print(f"Alpha elegido en T2: {alpha_t2:.4f}")
print(f"Alpha reproducido con GridSearchCV: {alpha_recalculado:.4f} (coincide: {coincide})")
print(f"R2 medio de validación cruzada (alpha elegido): {r2_cv_alpha:.4f}")
print("----------------------------------------------------")
print("Modelo 1) StandardScaler + Ridge (alpha elegido):")
print(f"  R2 prueba  = {r2_prueba_ridge:.4f}")
print(f"  RMSE prueba = {rmse_prueba_ridge:.4f}")
print("Modelo 2) StandardScaler + LinearRegression (sin regularizar):")
print(f"  R2 prueba  = {r2_prueba_linreg:.4f}")
print(f"  RMSE prueba = {rmse_prueba_linreg:.4f}")
print("----------------------------------------------------")
print(f"Diferencia R2 (Ridge - Lineal): {dif_r2:.4f}")
print(f"Diferencia RMSE (Ridge - Lineal): {dif_rmse:.4f}")
print(f"Mejor modelo en prueba (por R2): {mejor_modelo}")
print("Resultados guardados en resultados.json")


=== T3: Evaluación única sobre el conjunto de prueba ===
Entrenamiento: 331 ejemplos | Prueba: 111 ejemplos (test_size=0.25, random_state=7)
La prueba se usa una sola vez, con el alpha ya elegido en T2 (solo entrenamiento/CV).
----------------------------------------------------
Alpha elegido en T2: 0.3162
Alpha reproducido con GridSearchCV: 0.3162 (coincide: True)
R2 medio de validación cruzada (alpha elegido): 0.4724
----------------------------------------------------
Modelo 1) StandardScaler + Ridge (alpha elegido):
  R2 prueba  = 0.5037
  RMSE prueba = 54.3838
Modelo 2) StandardScaler + LinearRegression (sin regularizar):
  R2 prueba  = 0.5028
  RMSE prueba = 54.4331
----------------------------------------------------
Diferencia R2 (Ridge - Lineal): 0.0009
Diferencia RMSE (Ridge - Lineal): -0.0493
Mejor modelo en prueba (por R2): Ridge
Resultados guardados en resultados.json


Con el alpha ya fijado en 0.3162, evalué una sola vez sobre los 111 ejemplos de prueba. Ridge obtuvo R² = 0.5037 y RMSE = 54.3838. La regresión lineal sin regularizar, con el mismo escalado, obtuvo sobre la misma prueba R² = 0.5028 y RMSE = 54.4331. La diferencia es mínima (0.0009 de R² y 0.0493 de RMSE a favor de Ridge) y encaja con lo visto en la Parte 2: la curva de validación cruzada es casi plana para alphas pequeños, así que en estos datos la regularización apenas cambia al modelo lineal.

## Parte 4 — Figura

**Aviso: esta parte no se pudo completar.** La figura pedida —el R² medio de validación cruzada contra alpha, con el eje x en escala logarítmica mediante plt.xscale('log')— no llegó a generarse: el paso de graficación con matplotlib no produjo ninguna imagen y no hay ningún archivo de figura disponible para insertar en el notebook. La Parte 4 queda pendiente de generar y ejecutar esa celda de graficación. Lo que sí está calculado y a la vista son los datos de esa figura: los 13 R² medios por alpha que salen en la tabla de la Parte 2. Con ellos, la curva se vería casi plana en torno a 0.4724 entre alpha = 0.001 y alpha = 0.3162, alcanzaría su máximo en alpha = 0.3162 (R² = 0.4724) y bajaría después de forma clara: 0.4698 en alpha = 31.6228, 0.4570 en alpha = 100, 0.3995 en alpha = 316.2278 y 0.2790 en alpha = 1000.

## Parte 5 — Pregunta conceptual

Elegir alpha mirando el R² de prueba daría una estimación optimista porque la prueba dejaría de ser datos no vistos. Al probar 13 alphas y quedarme con el de mayor R² sobre esos mismos 111 ejemplos, tomo el máximo de 13 estimaciones ruidosas sobre una sola muestra: el alpha ganador suele ser, en parte, el que mejor se acomodó al ruido concreto de esos puntos, no el que generaliza mejor. Esa decisión mete información de la prueba en la construcción del modelo, y el número que luego reporto sobre esa misma prueba ya no estima el desempeño con datos nuevos, sino con datos que influyeron en una elección. El efecto se acumula: cada decisión adicional tomada mirando la prueba infla un poco más la estimación, y para recuperar un número honesto haría falta un conjunto de datos fresco.

En mis resultados, el R² medio de validación cruzada con el alpha elegido fue 0.4724 y el R² de prueba fue 0.5037, es decir, el de prueba queda por encima del de CV. Esa diferencia no significa que la validación cruzada estuviera sesgada ni que hubiera fuga: son dos estimaciones del mismo desempeño de generalización sobre muestras distintas. La de CV promedia 5 folds ajustados dentro de los 331 ejemplos de entrenamiento; la de prueba es una sola estimación sobre 111 ejemplos. Con una desviación estándar entre folds de 0.0709 y R² por fold que van de 0.3874 a 0.5643, una brecha de ese tamaño cabe de sobra dentro de esa variabilidad. De hecho, aquí la prueba salió más alta que la CV, lo que refuerza el punto: una sola división puede moverse en cualquier dirección. Por eso el alpha se eligió solo con validación cruzada sobre entrenamiento y la prueba se usó una única vez, ya con el alpha decidido.